In [6]:
import torch
import torch.nn as nn

# 1. Hyperparameters & Dimensions
input_size = 4    # Size of vector for x_t (e.g., word embedding size)
hidden_size = 5   # Size of memory vector h_t
output_size = 2   # Size of output vector y_t
seq_len = 3       # Sequence length T (time steps: t=1, t=2, t=3)

# 2. Define standard PyTorch RNN Cell
# An RNNCell represents a single unrolled step of an RNN
rnn_cell = nn.RNNCell(input_size, hidden_size)
output_layer = nn.Linear(hidden_size, output_size)
criterion = nn.MSELoss()

# 3. Dummy Sequence Data
# Input shape: (sequence_length, batch_size, input_size)
x_sequence = torch.randn(seq_len, 1, input_size)
# Target labels for each time step: (sequence_length, batch_size, output_size)
targets = torch.randn(seq_len, 1, output_size)

# Initialize h_0 (hidden state at t=0) with zeros
h_t = torch.zeros(1, hidden_size)

total_loss = 0

print("--- FORWARD PASS (UNROLLING THROUGH TIME) ---")
# Manually unrolling the loop over time steps t = 1..T
for t in range(seq_len):
    x_t = x_sequence[t]

    # Recurrent Step: h_t = tanh(W_hh * h_{t-1} + W_xh * x_t + b_h)
    h_t = rnn_cell(x_t, h_t)

    # Output Step: y_hat_t = W_hy * h_t + b_y
    y_hat_t = output_layer(h_t)

    # Step Loss L_t
    loss_t = criterion(y_hat_t, targets[t])
    total_loss += loss_t

    print(f"Time Step t={t+1}:")
    print(f"  Input x_t shape: {list(x_t.shape)}")
    print(f"  Hidden State h_{t+1} sample values: {h_t[0, :3].detach().numpy()}...")
    print(f"  Step Loss L_{t+1}: {loss_t.item():.4f}\n")

print(f"Total Sequence Loss L: {total_loss.item():.4f}")

# 4. BACKPROPAGATION THROUGH TIME (BPTT)
print("\n--- BACKWARD PASS (BPTT) ---")
# Compute gradients back through the entire unrolled computation graph
total_loss.backward()

# Inspect gradients of the shared hidden-to-hidden weight matrix (W_hh)
w_hh_grad = rnn_cell.weight_hh.grad
print("Gradient computed for shared W_hh matrix:")
print(w_hh_grad)

ModuleNotFoundError: No module named 'torch'

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Prepare Training Text & Vocabulary
text = "hello world! learning rnn character prediction."
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# 2. One-Hot Encoding Helper Function
def string_to_one_hot(seq_str):
    tensor = torch.zeros(len(seq_str), 1, vocab_size)
    for t, char in enumerate(seq_str):
        tensor[t][0][char_to_idx[char]] = 1.0
    return tensor

# 3. Define Character-Level RNN Model
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn_cell = nn.RNNCell(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x_t, h_t):
        h_t = self.rnn_cell(x_t, h_t)
        output = self.fc(h_t)
        return output, h_t

    def init_hidden(self):
        return torch.zeros(1, self.hidden_size)

# 4. Initialize Model, Loss, and Optimizer
hidden_size = 32
model = CharRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training inputs (X) and targets (Y shift by 1 step)
# Input: "hello world! learning rnn character prediction"
# Target: "ello world! learning rnn character prediction."
input_text = text[:-1]
target_text = text[1:]

inputs = string_to_one_hot(input_text)
targets = torch.tensor([char_to_idx[ch] for ch in target_text])

# 5. Training Loop (BPTT across sequence steps)
epochs = 200
print("--- TRAINING CHAR-LEVEL RNN ---")

for epoch in range(1, epochs + 1):
    h_t = model.init_hidden()
    optimizer.zero_grad()
    loss = 0

    # Unroll network across input sequence
    for t in range(len(input_text)):
        x_t = inputs[t]  # One-hot vector at time step t
        y_hat_t, h_t = model(x_t, h_t)

        # Accumulate Loss for target character at step t
        loss += criterion(y_hat_t, targets[t].unsqueeze(0))

    # Backpropagation Through Time
    loss.backward()
    optimizer.step()

    if epoch % 40 == 0:
        print(f"Epoch {epoch}/{epochs} | Total Loss: {loss.item():.4f}")

# 6. Inference Function (Generating Text from a Prompt)
def generate_text(model, start_str, generate_len=30):
    model.eval()
    h_t = model.init_hidden()
    generated = start_str

    # Warm up hidden state with start string
    for char in start_str[:-1]:
        x_t = string_to_one_hot(char)[0]
        _, h_t = model(x_t, h_t)

    # Generate new characters step-by-step
    current_char = start_str[-1]
    for _ in range(generate_len):
        x_t = string_to_one_hot(current_char)[0]
        y_hat_t, h_t = model(x_t, h_t)

        # Get highest probability predicted character
        predicted_idx = torch.argmax(y_hat_t, dim=1).item()
        next_char = idx_to_char[predicted_idx]

        generated += next_char
        current_char = next_char

    return generated

# Test Text Generation
print("\n--- TEXT GENERATION RESULTS ---")
print("Prompt: 'hel'")
print("Result:", generate_text(model, start_str="hel", generate_len=34))

--- TRAINING CHAR-LEVEL RNN ---
Epoch 40/200 | Total Loss: 7.1082
Epoch 80/200 | Total Loss: 0.6529
Epoch 120/200 | Total Loss: 0.3394
Epoch 160/200 | Total Loss: 0.2225
Epoch 200/200 | Total Loss: 0.1592

--- TEXT GENERATION RESULTS ---
Prompt: 'hel'
Result: hello world! learning rnn character p
